# CNN v3 — Expanded Training + Better Calibration

Changes: all sessions from non-interday train subjects, 50 epochs, LogReg C=0.1 for calibration.

In [1]:
import sys, math
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path: sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np, pandas as pd
import torch, torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.linear_model import LogisticRegression

from config import RANDOM_SEED, N_CLASSES, MODELS_DIR, get_device
from src.experiment_runner import (
    get_splits, load_and_norm,
    run_zero_shot, run_calibration, print_comparison,
)
from src.evaluation import measure_latency, print_latency

torch.manual_seed(RANDOM_SEED); np.random.seed(RANDOM_SEED)
DEVICE = get_device()
splits = get_splits()
print(f'Device: {DEVICE}')

Device: mps


In [2]:
class StandardCNN(nn.Module):
    def __init__(self, n_classes=N_CLASSES):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1,32,3,padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32,64,3,padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64,128,3,padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.AdaptiveAvgPool2d(1),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Dropout(0.3), nn.Linear(128,64), nn.ReLU(),
            nn.Dropout(0.3), nn.Linear(64,n_classes),
        )
    def forward(self,x): return self.classifier(self.features(x))
    def extract_feat(self,x):
        with torch.no_grad(): return nn.Flatten()(self.features(x))

print(f'Params: {sum(p.numel() for p in StandardCNN().parameters()):,}')

Params: 101,831


In [3]:
train_combined = pd.concat([splits['train_df'], splits['s5_train']])
X_train, y_train, norm_stats = load_and_norm(train_combined, verbose=True)
print(f'Train: {X_train.shape}')

loader = DataLoader(TensorDataset(
    torch.from_numpy(X_train).float().unsqueeze(1),
    torch.from_numpy(y_train).long()
), batch_size=256, shuffle=True)

model = StandardCNN().to(DEVICE)
opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=1e-3, epochs=50, steps_per_epoch=len(loader))
crit = nn.CrossEntropyLoss(label_smoothing=0.1)

for ep in range(50):
    model.train()
    ls,c,t = 0,0,0
    for xb,yb in loader:
        xb,yb = xb.to(DEVICE),yb.to(DEVICE)
        out = model(xb); loss = crit(out,yb)
        opt.zero_grad(); loss.backward(); opt.step(); sched.step()
        ls += loss.item()*xb.size(0); c += (out.argmax(1)==yb).sum().item(); t += xb.size(0)
    if (ep+1)%10==0 or ep==0:
        print(f'Epoch {ep+1:2d}/50 — loss:{ls/t:.4f} acc:{c/t:.4f}')

torch.save(model.state_dict(), MODELS_DIR / 'cnn_v3.pt')
print('Saved.')

Loading windows: 100%|██████████| 9021/9021 [00:04<00:00, 1900.03it/s]


Train: (1030712, 8, 50)
Epoch  1/50 — loss:1.6318 acc:0.4027
Epoch 10/50 — loss:1.1808 acc:0.6559
Epoch 20/50 — loss:1.0935 acc:0.7007
Epoch 30/50 — loss:1.0544 acc:0.7215
Epoch 40/50 — loss:1.0132 acc:0.7434
Epoch 50/50 — loss:0.9890 acc:0.7560
Saved.


In [4]:
@torch.no_grad()
def cnn_predict(X):
    model.eval()
    Xt = torch.from_numpy(X).float().unsqueeze(1)
    loader = DataLoader(TensorDataset(Xt), batch_size=512, shuffle=False)
    return np.concatenate([model(xb[0].to(DEVICE)).argmax(1).cpu().numpy() for xb in loader])

@torch.no_grad()
def cnn_features(X):
    model.eval()
    Xt = torch.from_numpy(X).float().unsqueeze(1)
    loader = DataLoader(TensorDataset(Xt), batch_size=512, shuffle=False)
    return np.concatenate([model.extract_feat(xb[0].to(DEVICE)).cpu().numpy() for xb in loader])

def cnn_finetune(X_cal, y_cal):
    F = cnn_features(X_cal)
    clf = LogisticRegression(max_iter=2000, random_state=RANDOM_SEED, C=0.1)
    clf.fit(F, y_cal)
    def predict_ft(X): return clf.predict(cnn_features(X))
    return predict_ft

print('Option B — Zero-shot:')
zero_results = run_zero_shot(cnn_predict, splits, norm_stats)
print('\nOption A — Calibration:')
cal_results = run_calibration(cnn_predict, cnn_finetune, splits, norm_stats)

Option B — Zero-shot:
  S1 zero-shot: 0.6131
  S2 zero-shot: 0.5715
  S3 zero-shot: 0.5752
  S4 zero-shot: 0.6704
  S5 zero-shot: 0.7989

Option A — Calibration:
  S1 calibrated: 0.6824
  S2 calibrated: 0.7009
  S3 calibrated: 0.7821
  S4 calibrated: 0.7394
  S5 calibrated: 0.8285


In [5]:
model.eval()
s = torch.randn(1,1,8,50).to(DEVICE)
for _ in range(10): _ = model(s)
if DEVICE.type=='mps': torch.mps.synchronize()
def cnn_single(x):
    xt = torch.from_numpy(x).float().unsqueeze(1).to(DEVICE)
    with torch.no_grad(): out = model(xt)
    if DEVICE.type=='mps': torch.mps.synchronize()
    return out.argmax(1).cpu().numpy()
latency = measure_latency(cnn_single, X_train[:1], n_runs=500)
print_latency(latency, 'CNN')
print_comparison(zero_results, cal_results, name='CNN (v3)')


Latency — CNN
  Mean:   4.78 ms
  Median: 3.49 ms
  P95:    10.24 ms
  <300ms: ✓

  CNN (v3) — RESULTS
Scenario        Zero-shot   Calibrated        Δ
-------------------------------------------------------
S1                61.31%       68.24%    +6.93%
S2                57.15%       70.09%   +12.94%
S3                57.52%       78.21%   +20.68%
S4                67.04%       73.94%    +6.90%
S5                79.89%       82.85%    +2.96%
